# Session 26: Image Classification with Neural Networks
### Duration: ~2-2.5 Hours

---

## Session Goal
In Sessions 24 and 25, we learned what neural networks are and where different types of neural networks are used.

In this session, we take the next step:

> Use a neural network to classify images.

We will use the famous built-in **Digits dataset** from scikit-learn. It contains small handwritten digit images from 0 to 9.

## Why This Dataset?
This dataset is perfect for a first image-classification class because:

- It is real image data.
- It is built into scikit-learn, so no internet download is needed.
- Images are small: `8 x 8` pixels.
- The task is easy to understand: predict the digit.
- We can use the same tools students already know.

## Big Idea
An image is just a grid of numbers.

For example:

```text
8 x 8 image = 64 pixel values
```

A neural network can learn patterns from those pixel values.

---
## 1. From Tabular Data to Image Data

Earlier, our datasets looked like this:

```text
row = one customer / one flower / one house
columns = features
```

For images:

```text
one image = grid of pixels
one pixel = number representing brightness or color
```

In this dataset:

```text
0 = dark pixel
16 = bright pixel
```

Each image has shape:

```text
8 rows x 8 columns
```

For a basic neural network, we flatten the image:

```text
8 x 8 grid -> 64 input numbers
```

This is not yet a CNN. Today we use a simple feedforward neural network first.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')

RANDOM_STATE = 42
OUTPUT_DIR = Path('session26_artifacts')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Environment ready.')

---
## 2. Load the Digits Dataset

The dataset contains:

- `images`: original 8 x 8 image grids
- `data`: flattened 64-value feature rows
- `target`: correct digit label from 0 to 9

In [ ]:
digits = load_digits()

images = digits.images
X = digits.data
y = digits.target
class_names = [str(i) for i in digits.target_names]

print('Images shape:', images.shape)
print('Flattened data shape:', X.shape)
print('Target shape:', y.shape)
print('Classes:', class_names)

In [ ]:
# Put the flattened pixel data into a DataFrame for familiar inspection.
pixel_columns = [f'pixel_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=pixel_columns)
df['digit'] = y

print('Dataset shape:', df.shape)
df.head()

---
## 3. Visualize the Images

Before modeling, always look at the data.

These are small grayscale images. Each image is only 8 x 8 pixels, but humans can still recognize many digits.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
axes = axes.ravel()

for digit in range(10):
    index = np.where(y == digit)[0][0]
    axes[digit].imshow(images[index], cmap='gray_r')
    axes[digit].set_title(f'Digit: {digit}')
    axes[digit].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
axes = axes.ravel()

for ax, image, label in zip(axes, images[:32], y[:32]):
    ax.imshow(image, cmap='gray_r')
    ax.set_title(str(label))
    ax.axis('off')

plt.suptitle('Sample Handwritten Digits', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Inspect Pixel Values

Let us inspect one image as numbers.

This is the key idea for image classification:

> The computer does not see a digit the way we do. It sees numbers.

In [ ]:
sample_index = 0
sample_image = images[sample_index]

print('Correct label:', y[sample_index])
print('Image shape:', sample_image.shape)
print('\nPixel grid:')
print(sample_image)

plt.figure(figsize=(4, 4))
sns.heatmap(sample_image, annot=True, fmt='.0f', cmap='gray_r', cbar=False, square=True)
plt.title(f'Pixel Values for Digit {y[sample_index]}')
plt.xlabel('Column')
plt.ylabel('Row')
plt.show()

---
## 5. Class Distribution

A good classification dataset should be checked for class balance.

If one digit appears much more than the others, accuracy may become misleading.

In [ ]:
class_counts = pd.Series(y, name='digit').value_counts().sort_index()

plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values)
plt.title('Number of Images Per Digit')
plt.xlabel('Digit')
plt.ylabel('Image Count')
plt.tight_layout()
plt.show()

class_counts

---
## 6. Train-Test Split

We split the image data into training and test sets.

We use `stratify=y` to keep the digit distribution similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Training class counts:', np.bincount(y_train))
print('Test class counts:', np.bincount(y_test))

---
## 7. Baseline Model: Logistic Regression

Before using a neural network, we train a familiar classical ML model.

This gives us a baseline.

For this dataset, Logistic Regression can perform surprisingly well.

In [ ]:
logistic_model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_accuracy = accuracy_score(y_test, logistic_pred)

print('Logistic Regression accuracy:', round(logistic_accuracy, 4))
print('\nClassification report:')
print(classification_report(y_test, logistic_pred, target_names=class_names))

---
## 8. First Image Neural Network

Now we train a basic feedforward neural network using `MLPClassifier`.

The input is 64 pixel values.

Network idea:

```text
64 pixel inputs -> hidden layer -> 10 digit outputs
```

The output has 10 classes because the digits are 0 through 9.

In [ ]:
image_nn = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(64,),
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        max_iter=300,
        random_state=RANDOM_STATE,
    )),
])

image_nn.fit(X_train, y_train)

nn_pred = image_nn.predict(X_test)
nn_accuracy = accuracy_score(y_test, nn_pred)

print('Neural Network accuracy:', round(nn_accuracy, 4))
print('\nClassification report:')
print(classification_report(y_test, nn_pred, target_names=class_names))

---
## 9. Training Loss Curve

The neural network learns by reducing loss.

If the loss curve goes down, the model is learning from the training data.

In [ ]:
mlp = image_nn.named_steps['classifier']

plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_)
plt.title('Neural Network Training Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.tight_layout()
plt.show()

print('Training iterations:', mlp.n_iter_)
print('Final loss:', round(mlp.loss_, 4))

---
## 10. Confusion Matrix

The confusion matrix helps us see which digits the model confuses.

For example, handwritten `8` and `9` can sometimes look similar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ConfusionMatrixDisplay.from_estimator(
    logistic_model,
    X_test,
    y_test,
    display_labels=class_names,
    cmap='Blues',
    ax=axes[0],
)
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay.from_estimator(
    image_nn,
    X_test,
    y_test,
    display_labels=class_names,
    cmap='Greens',
    ax=axes[1],
)
axes[1].set_title('Neural Network')

plt.tight_layout()
plt.show()

---
## 11. Compare Models

A neural network is not automatically better than every classical ML model.

We compare both models on test accuracy.

In [ ]:
comparison = pd.DataFrame({
    'model': ['Logistic Regression', 'Basic Neural Network'],
    'test_accuracy': [logistic_accuracy, nn_accuracy],
})

comparison.round(4)

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=comparison, x='test_accuracy', y='model')
plt.xlim(0, 1.05)
plt.title('Image Classification Model Comparison')
plt.xlabel('Test Accuracy')
plt.ylabel('')
plt.tight_layout()
plt.show()

---
## 12. Make Predictions on Test Images

Let us view some test images with their predicted labels.

Green titles mean correct predictions.
Red titles mean incorrect predictions.

In [ ]:
# Recover the original image grids for test examples.
_, images_test, _, y_test_for_images = train_test_split(
    images,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

predictions = image_nn.predict(X_test)

fig, axes = plt.subplots(4, 5, figsize=(10, 8))
axes = axes.ravel()

for ax, image, actual, pred in zip(axes, images_test[:20], y_test_for_images[:20], predictions[:20]):
    ax.imshow(image, cmap='gray_r')
    color = 'green' if actual == pred else 'red'
    ax.set_title(f'Actual: {actual} | Pred: {pred}', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 13. View Incorrect Predictions

Mistakes are useful.

They help us understand where the model struggles.

In [ ]:
wrong_indices = np.where(predictions != y_test)[0]
print('Number of incorrect predictions:', len(wrong_indices))

if len(wrong_indices) > 0:
    n_show = min(12, len(wrong_indices))
    fig, axes = plt.subplots(3, 4, figsize=(10, 7))
    axes = axes.ravel()

    for ax, wrong_index in zip(axes, wrong_indices[:n_show]):
        ax.imshow(images_test[wrong_index], cmap='gray_r')
        ax.set_title(
            f"Actual: {y_test[wrong_index]} | Pred: {predictions[wrong_index]}",
            color='red',
        )
        ax.axis('off')

    for ax in axes[n_show:]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('No mistakes found on this test split.')

---
## 14. Prediction Probabilities

The model does not only predict a class.

It can also give probabilities for each digit.

This tells us how confident the model is.

In [ ]:
example_number = 3
example_image = images_test[example_number]
example_features = X_test[example_number].reshape(1, -1)

probabilities = image_nn.predict_proba(example_features)[0]
prob_table = pd.DataFrame({
    'digit': class_names,
    'probability': probabilities,
}).sort_values('probability', ascending=False)

plt.figure(figsize=(4, 4))
plt.imshow(example_image, cmap='gray_r')
plt.title(f'Actual Digit: {y_test[example_number]}')
plt.axis('off')
plt.show()

prob_table.round(4)

---
## 15. Optional Visualization: 2D PCA Projection

Each image has 64 pixel features.

That is hard to visualize directly.

PCA can reduce the data to 2 dimensions so we can make a simple scatter plot.

This does not train the neural network. It only helps us visualize the dataset.

In [ ]:
scaled_X = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(scaled_X)

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'digit': y.astype(str),
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='digit', palette='tab10', s=35, alpha=0.8)
plt.title('Digits Dataset Projected to 2D with PCA')
plt.legend(title='Digit', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('Explained variance by first 2 components:', pca.explained_variance_ratio_.round(3))

---
## 16. Experiment: Different Hidden Layer Sizes

The network architecture affects learning.

Let us compare a few simple neural networks.

In [ ]:
network_configs = {
    'Small: 32 neurons': (32,),
    'Medium: 64 neurons': (64,),
    'Two layers: 64 and 32': (64, 32),
}

experiment_results = []
trained_networks = {}

for name, hidden_layers in network_configs.items():
    model = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('classifier', MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation='relu',
            solver='adam',
            learning_rate_init=0.001,
            max_iter=300,
            random_state=RANDOM_STATE,
        )),
    ])
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    classifier = model.named_steps['classifier']
    trained_networks[name] = model
    experiment_results.append({
        'network': name,
        'hidden_layer_sizes': hidden_layers,
        'test_accuracy': accuracy_score(y_test, preds),
        'final_loss': classifier.loss_,
        'iterations': classifier.n_iter_,
    })

experiment_df = pd.DataFrame(experiment_results).sort_values('test_accuracy', ascending=False)
experiment_df.round(4)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=experiment_df, x='test_accuracy', y='network')
plt.xlim(0, 1.05)
plt.title('Neural Network Architecture Experiment')
plt.xlabel('Test Accuracy')
plt.ylabel('')
plt.tight_layout()
plt.show()

---
## 17. Why CNNs Come Next

Today we flattened each image:

```text
8 x 8 image -> 64 numbers
```

This works for small images, but it loses some spatial structure.

A CNN is better for images because it looks at local pixel patterns.

CNNs can learn:

```text
edges -> curves -> parts -> objects
```

So the next natural topic is:

```text
Convolutional Neural Networks for Image Classification
```

---
## 18. Save Model and Outputs

A trained image classifier can be saved and reused later.

In [ ]:
best_network_name = experiment_df.iloc[0]['network']
best_network = trained_networks[best_network_name]

model_path = OUTPUT_DIR / 'digits_image_neural_network.joblib'
comparison_path = OUTPUT_DIR / 'image_model_comparison.csv'
experiment_path = OUTPUT_DIR / 'image_network_experiment.csv'
probability_path = OUTPUT_DIR / 'example_prediction_probabilities.csv'

dump(best_network, model_path)
comparison.to_csv(comparison_path, index=False)
experiment_df.to_csv(experiment_path, index=False)
prob_table.to_csv(probability_path, index=False)

print('Saved best neural network:', model_path)
print('Best network:', best_network_name)
print('Saved model comparison:', comparison_path)
print('Saved architecture experiment:', experiment_path)
print('Saved example probabilities:', probability_path)

---
## 19. Class Discussion Questions

Use these questions during class:

1. What is a pixel?
2. What does an image look like to a computer?
3. Why does an 8 x 8 image become 64 input values?
4. Why do we scale pixel features before training?
5. What does the loss curve tell us?
6. Which digits were confused most often?
7. Why might handwritten digits be difficult for a model?
8. Was the neural network always better than Logistic Regression?
9. What information do prediction probabilities give us?
10. Why are CNNs better suited for larger image problems?

---

## Final Summary

In this session, we learned that images are numerical data.

A grayscale image is a grid of pixel values.

For this first image-classification model, we used:

```text
8 x 8 image -> flatten to 64 numbers -> neural network -> digit prediction
```

Important ideas:

- Image data can be represented as arrays.
- Neural networks can learn patterns from pixels.
- Scaling helps neural networks train.
- Confusion matrices reveal which classes are difficult.
- Wrong predictions are useful for understanding model limitations.
- CNNs are the next step because they preserve image structure better than basic MLPs.

Next recommended session:

```text
Session 27: Convolutional Neural Networks
```